# Phase 7 — GRPO fine-tune of Qwen2.5-3B on TokenEfficiencyEnv

Trains the model to answer questions correctly **with fewer tokens** by giving it our 6-component reward (correctness 55% + efficiency 15% + self-assessment 15% + redundancy/keyword/format 5% each), and lets GRPO do the rest.

**Reading order**: each cell has a one-line purpose comment up top. Run them in order. The whole notebook runs end-to-end on a single 16GB+ GPU in ~1–2h with the default 300-step config; bump `max_steps` once you trust it.

**What you'll get out**: an `outputs/grpo_qwen2.5_3b/` directory with the LoRA adapter + a `metrics.json`, plus before/after plots showing token usage going down without correctness dropping.

**What you need set**:
* `HF_TOKEN` env var (for the judge — used both during training and eval).
* `pip install -r ../requirements-train.txt` from the repo root.
* CUDA-capable GPU recommended; 16GB enough with LoRA + bf16.

In [ ]:
# 1) Path setup — make `training` and `token_efficiency_env` importable when
#    you launch jupyter from anywhere.
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT =", REPO_ROOT)
print("HF_TOKEN  =", "set" if os.environ.get("HF_TOKEN") else "NOT SET — judge will fall back to keyword scoring")

In [ ]:
# 2) Config — single source of truth. Edit values here, NOT in cells below.
from training import TrainingConfig

config = TrainingConfig(
    # Defaults are fine; uncomment to override:
    # max_steps=100,           # quick smoke run
    # judge_backend="keyword", # offline dry-run, zero API calls
    # num_generations=4,       # smaller GPU
)
print(config.describe())

In [ ]:
# 3) Train/holdout split — print so you can SEE what the model never saw.
from training import describe_split, holdout_prompts, train_prompts

print(describe_split())
print(f"\nTraining on {len(train_prompts())} prompts, evaluating on {len(holdout_prompts())}.")

## 4) (Optional) Server pool

Skip this cell if `config.reward_backend == "in_process"` (the default). Only run it if you flipped to `"ws"` to stress-test the deployed server path.

In [ ]:
# 4) Server pool — only spawned if reward_backend == "ws".
pool = None
if config.reward_backend == "ws":
    from training import ServerPool
    os.environ["JUDGE_BACKEND"] = config.judge_backend  # workers inherit this
    pool = ServerPool(
        num_envs=config.num_env_servers,
        base_port=config.base_port,
        judge_backend=config.judge_backend,
    ).start()
    print("Pool ready:", pool.urls)
else:
    print("reward_backend = in_process; skipping server pool.")

In [ ]:
# 5) Reward function — wires the env into TRL's reward_funcs API.
#    The same JUDGE_BACKEND env var is honoured by the in-process env.
from training import RewardLog, build_reward_func

os.environ["JUDGE_BACKEND"] = config.judge_backend
reward_log = RewardLog()
reward_func, reward_adapter = build_reward_func(config, log=reward_log)

_test_rewards = reward_func(
    ["What is the capital of France?"] * 2,
    ["<budget>3</budget><answer>Paris.</answer>",
     "<budget>3</budget><answer>London.</answer>"],
)
print("smoke-test rewards (Paris vs London):", _test_rewards)
assert _test_rewards[0] > _test_rewards[1], "sanity check: Paris should out-score London"

In [ ]:
# 6) Tokenizer + base model + LoRA. bf16 keeps memory in check on 16GB cards.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Model loaded on {model.device} | dtype={dtype}")

if config.use_lora:
    from peft import LoraConfig, get_peft_model
    lora_cfg = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=config.lora_target_modules,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

In [ ]:
# 7) Build the training dataset. GRPO eats {"prompt": str} rows.
#    The system prompt is identical to the one in eval_baseline_vs_trained.py
#    so before/after numbers stay comparable.
from datasets import Dataset
from notebooks.eval_baseline_vs_trained import SYSTEM_PROMPT

def _to_chat(question: str) -> str:
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

train_ds = Dataset.from_list([
    {"prompt": _to_chat(t["prompt"]), "raw_prompt": t["prompt"]}
    for t in train_prompts()
])
print(train_ds)
print("\nFirst rendered prompt:\n", train_ds[0]["prompt"][:400], "\n...")

In [ ]:
# 8) Reward-function shim that maps TRL's CHAT-rendered prompt back to the
#    raw question, so the env's prompt → task lookup still works.
_chat_to_raw = {row["prompt"]: row["raw_prompt"] for row in train_ds}

def trl_reward(prompts, completions, **kw):
    raw_prompts = [_chat_to_raw.get(p, p) for p in prompts]
    return reward_func(raw_prompts, completions, **kw)

In [ ]:
# 9) Baseline eval BEFORE training — establishes the bar to beat.
from notebooks.eval_baseline_vs_trained import evaluate, print_summary

baseline = evaluate(
    model, tokenizer, holdout_prompts(),
    label="baseline (untrained Qwen2.5-3B)",
    max_new_tokens=config.max_completion_length,
    temperature=config.eval_temperature,
    samples_per_prompt=config.eval_samples_per_prompt,
)
print_summary(baseline)

In [ ]:
# 10) GRPO trainer setup. Most knobs come straight from `config`.
from trl import GRPOConfig, GRPOTrainer

grpo_cfg = GRPOConfig(
    output_dir=config.output_dir,
    num_train_epochs=1,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    num_generations=config.num_generations,
    max_prompt_length=config.max_prompt_length,
    max_completion_length=config.max_completion_length,
    temperature=config.temperature,
    seed=config.seed,
    logging_steps=5,
    save_steps=max(50, config.max_steps // 6),
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    report_to=[],  # no wandb/tensorboard by default; flip on if you want
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_cfg,
    train_dataset=train_ds,
    reward_funcs=[trl_reward],
)
print("GRPOTrainer ready.")

In [ ]:
# 11) TRAIN. This is the long-running cell. Reward log is filled as it goes.
trainer.train()
print("\nTraining done.\n")
trainer.save_model(config.output_dir)
print(f"Checkpoint saved to: {config.output_dir}")

In [ ]:
# 12) Trained-model eval — same held-out prompts, same harness.
trained = evaluate(
    model, tokenizer, holdout_prompts(),
    label="trained (LoRA adapter applied)",
    max_new_tokens=config.max_completion_length,
    temperature=config.eval_temperature,
    samples_per_prompt=config.eval_samples_per_prompt,
)
print_summary(trained)

In [ ]:
# 13) Side-by-side comparison + plots.
import json
import pandas as pd
import matplotlib.pyplot as plt
from notebooks.eval_baseline_vs_trained import summary_to_dict

comp = pd.DataFrame({
    "metric": ["reward", "correctness", "tokens_used", "overshoot_rate", "cliff_rate"],
    "baseline": [baseline.mean_reward, baseline.mean_correctness,
                 baseline.mean_tokens_used, baseline.overshoot_rate,
                 baseline.cliff_rate],
    "trained":  [trained.mean_reward, trained.mean_correctness,
                 trained.mean_tokens_used, trained.overshoot_rate,
                 trained.cliff_rate],
})
comp["delta"] = comp["trained"] - comp["baseline"]
display(comp)

rewards_df = pd.DataFrame(reward_log.to_records())
if not rewards_df.empty:
    rewards_df["step"] = rewards_df.index // max(config.num_generations, 1)
    rolling = rewards_df.groupby("step")["reward"].mean().rolling(25, min_periods=1).mean()
    plt.figure(figsize=(10, 4))
    plt.plot(rolling.index, rolling.values)
    plt.xlabel("GRPO step")
    plt.ylabel("reward (rolling mean over 25 steps)")
    plt.title("Training-time reward")
    plt.grid(True, alpha=0.3)
    plt.show()

metrics = {
    "config": config.describe(),
    "baseline": summary_to_dict(baseline),
    "trained":  summary_to_dict(trained),
}
Path(config.output_dir).mkdir(parents=True, exist_ok=True)
metrics_path = Path(config.output_dir) / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Wrote {metrics_path}")

In [ ]:
# 14) Cleanup — close adapter (no-op for in_process) and shut down the
#     server pool if we spawned one. Wrap in try so a crash above still
#     releases ports.
try:
    reward_adapter.close()
finally:
    if pool is not None:
        pool.shutdown()
        print("Server pool shut down.")
    else:
        print("Done.")